In [ ]:
import fitz  # pip install PyMuPDF
import logging
from typing import Optional, Union, List, Dict
import re
import numpy as np
import math
import os
import json
from difflib import SequenceMatcher

# Functions 


In [ ]:
def combine_to_json(
    paragraphs: List[str],
    highlights: List[Dict],
    sticky_notes: List[Dict],
    context_window: int = 20,
    logger: Optional[logging.Logger] = None
) -> Dict:
    """
    Combines paragraphs, highlights, and sticky notes into a structured JSON format.
    """
    if logger is None:
        logger = logging.getLogger(__name__)
        logger.addHandler(logging.NullHandler())

    def normalize(s: str) -> str:
        """Remove all whitespace for matching."""
        return re.sub(r"\s+", "", s or "")

    def build_index_map(text: str) -> List[int]:
        """Map indices in normalized text back to original text indices."""
        mapping = []
        for i, ch in enumerate(text):
            if not ch.isspace():
                mapping.append(i)
        return mapping

    def adjust_boundary(start: int, end: int, para: str, prov_cl: str, prov_cr: str):
        """
        Adjusts highlight start/end to include or exclude partial words based on prov_cl and prov_cr.
        Left: include full word if highlighted head covers >=50%, else drop head fragment.
        Right: include full word if tail covers >=50%, else drop tail fragment.
        """
        # Determine full word around start
        left_ws = para.rfind(' ', 0, start) + 1
        right_ws = para.find(' ', start)
        if right_ws < 0:
            right_ws = len(para)
        full_word = para[left_ws:right_ws]
        # head fragment from highlight boundary
        head = ''
        for i in range(start, end):
            if para[i].isspace():
                break
            head += para[i]
        # adjust left
        if full_word and head:
            if len(head) >= 0.5 * len(full_word):
                new_start = left_ws
            else:
                new_start = start + len(head)
        else:
            new_start = start

        # Determine full word around end
        left_ws_r = para.rfind(' ', 0, end) + 1
        right_ws_r = para.find(' ', end)
        if right_ws_r < 0:
            right_ws_r = len(para)
        full_word_r = para[left_ws_r:right_ws_r]
        # tail fragment before end
        tail = ''
        for i in range(end - 1, -1, -1):
            if para[i].isspace():
                break
            tail = para[i] + tail
        # adjust right
        if full_word_r and tail:
            if len(tail) >= 0.5 * len(full_word_r):
                new_end = right_ws_r
            else:
                new_end = end - len(tail)
        else:
            new_end = end

        return new_start, new_end

    output = {"paragraphs": []}
    per_par_annotations = [[] for _ in paragraphs]
    unmatched = []

    # Prepare unified annotations
    annotations = []
    for hl in highlights:
        annotations.append({
            "type": "highlight",
            "orig_text": hl.get("h_text", "").replace("\n", " "),
            "comment": hl.get("content", ""),
            "ctx_left_in": hl.get("context_left", ""),
            "ctx_right_in": hl.get("context_right", ""),
            "color": hl.get("color", ""),
            "used": False
        })
    for note in sticky_notes:
        annotations.append({
            "type": "note",
            "orig_line": note.get("line", ""),
            "orig_text": note.get("local-text", "").replace("\n", " "),
            "comment": note.get("comment", ""),
            "ctx_left_in": note.get("context_left", "").replace("\n", " "),
            "ctx_right_in": note.get("context_right", "").replace("\n", " "),
            "used": False
        })

    # Match
    for ann in annotations:
        matched = False
        for p_idx, para in enumerate(paragraphs):
            norm_para = normalize(para)
            idx_map = build_index_map(para)

            if ann["type"] == "highlight":
                prov_cl_norm = normalize(ann.get("ctx_left_in", ""))
                prov_cr_norm = normalize(ann.get("ctx_right_in", ""))
                tgt_norm = normalize(ann["orig_text"])
                pos_norm = norm_para.find(tgt_norm)
                if pos_norm < 0:
                    continue
                # Check if independent contexts exist
                if (not prov_cl_norm or prov_cl_norm in norm_para) and (not prov_cr_norm or prov_cr_norm in norm_para):
                    # Found all parts independently
                    start0 = idx_map[pos_norm]
                    end0 = idx_map[pos_norm + len(tgt_norm) - 1] + 1
                else:
                    # Strict match combinations
                    combos = []
                    if prov_cl_norm and prov_cr_norm:
                        combos = [prov_cl_norm + tgt_norm + prov_cr_norm,
                                  prov_cl_norm[:-1] + tgt_norm + prov_cr_norm,
                                  prov_cl_norm + tgt_norm + prov_cr_norm[1:],
                                  prov_cl_norm[:-1] + tgt_norm + prov_cr_norm[1:]]
                    elif prov_cl_norm:
                        combos = [prov_cl_norm + tgt_norm, prov_cl_norm[:-1] + tgt_norm]
                    elif prov_cr_norm:
                        combos = [tgt_norm + prov_cr_norm, tgt_norm + prov_cr_norm[1:]]
                    else:
                        combos = [tgt_norm]
                    found_combo = False
                    for combo in combos:
                        combo_pos = norm_para.find(combo)
                        if combo_pos >= 0:
                            # h_text segment within combo
                            inner_pos = combo.find(tgt_norm)
                            pos_norm = combo_pos + inner_pos
                            start0 = idx_map[pos_norm]
                            end0 = idx_map[pos_norm + len(tgt_norm) - 1] + 1
                            found_combo = True
                            break
                    if not found_combo:
                        continue
                # Adjust boundaries
                new_start, new_end = adjust_boundary(start0, end0, para, ann["ctx_left_in"], ann["ctx_right_in"])
                # Extract normalized adjusted span and recalc positions
                adjusted_norm = normalize(para[new_start:new_end])
                new_pos_norm = norm_para.find(adjusted_norm)
                if new_pos_norm < 0:
                    continue
                new_end_norm = new_pos_norm + len(adjusted_norm)
                # Finally extract context surroundings
                start, end = new_start, new_end
                cl = para[max(0, start - context_window):start]
                cr = para[end:end + context_window]
                text = para[start:end]
                entry = {"type": "highlight", "color": ann["color"], "text": text,
                         "comment": ann["comment"], "context_left": cl, "context_right": cr}

            else:
                # Note matching
                prov_cl = ann.get("ctx_left_in", "")
                prov_cr = ann.get("ctx_right_in", "")
                tgt_line_norm = normalize(ann["orig_line"])
                pos_norm = normalize(para).find(tgt_line_norm)
                if pos_norm < 0:
                    continue
                idx_map = build_index_map(para)
                ls = idx_map[pos_norm]
                le = idx_map[pos_norm + len(tgt_line_norm) - 1] + 1
                local_norm = normalize(ann.get("orig_text", ""))
                if local_norm:
                    sub = normalize(para[ls:le])
                    sub_pos = sub.find(local_norm)
                    if sub_pos >= 0:
                        start = idx_map[pos_norm + sub_pos]
                        end = idx_map[pos_norm + sub_pos + len(local_norm) - 1] + 1
                    else:
                        start = le
                        end = le
                else:
                    start = le
                    end = le
                # Extend contexts only if provided
                if prov_cl:
                    if len(prov_cl) < context_window:
                        extra_len = context_window - len(prov_cl)
                        extra_start = max(0, start - extra_len)
                        extra = para[extra_start:start]
                        new_cl = extra + prov_cl
                    else:
                        new_cl = prov_cl
                else:
                    new_cl = ""
                if prov_cr:
                    if len(prov_cr) < context_window:
                        extra_len = context_window - len(prov_cr)
                        extra = para[end:end + extra_len]
                        new_cr = prov_cr + extra
                    else:
                        new_cr = prov_cr
                else:
                    new_cr = ""
                entry = {"type": "note", "comment": ann["comment"], "context_left": new_cl, "context_right": new_cr}

            per_par_annotations[p_idx].append(entry)
            ann["used"] = True
            matched = True
            break
        if not matched:
            logger.error("NOT fatal: unmatched annotation: %s", ann)
            unmatched.append(ann)

    # Build output paragraphs
    for i, p in enumerate(paragraphs, 1):
        output["paragraphs"].append({"body": p,
                                      "annotations": per_par_annotations[i - 1]})
    if unmatched:
        logger.error("NOT fatal: unmatched annotations present")
        extra = []
        for ann in unmatched:
            if ann["type"] == "highlight":
                extra.append({"type": "highlight", "color": ann.get("color", ""),
                              "text": ann.get("orig_text", ""), "comment": ann.get("comment", ""),
                              "context_left": ann.get("ctx_left_in", ""),
                              "context_right": ann.get("ctx_right_in", "")})
            else:
                extra.append({"type": "note", "comment": ann.get("comment", ""),
                              "context_left": ann.get("ctx_left_in", ""),
                              "context_right": ann.get("ctx_right_in", "")})
        output["paragraphs"].append({"body": "", "annotations": extra})
    return output


In [ ]:
def paragraphize(
    text: str,
    logger: Optional[logging.Logger] = None,
    new_paragraph_symbol: str = '<NEW_PARAGRAPH>',
    new_page_symbol: str = '<NEW_PAGE>'
) -> List[str]:
    """
    Splits raw text into paragraphs by detecting paragraph symbols,
    handles page breaks, hyphenated line endings, excess newlines, and multiple spaces.

    Args:
        text: Input string potentially containing new_page_symbol markers.
        logger: Optional logger to report issues.
        new_paragraph_symbol: String used to mark paragraph boundaries.
        new_page_symbol: String in text indicating page breaks (removed).

    Returns:
        A list of cleaned paragraph strings.
    """
    # Merge hyphenated line endings (e.g., 'beginn-\ning' -> 'beginning').
    cleaned = re.sub(r'-\n\s*', '-', text) # Not going to remove the dashes...

    # Normalize all remaining newlines to spaces
    cleaned = re.sub(r'\n+', ' ', cleaned)

    # Remove page break symbols
    if new_page_symbol:
        cleaned = cleaned.replace(new_page_symbol, '')

    # Collapse multiple spaces (including those from removed page breaks)
    cleaned = re.sub(r' {2,}', ' ', cleaned)

    # Split on the paragraph symbol
    parts = cleaned.split(new_paragraph_symbol)

    # Strip whitespace and filter out empty strings
    paragraphs: List[str] = []
    for idx, part in enumerate(parts):
        para = part.strip()
        if para:
            paragraphs.append(para)
        else:
            if logger:
                logger.debug(f"Skipped empty paragraph at index {idx}")

    if not paragraphs and logger:
        logger.warning("No paragraphs detected (resulting list is empty)")

    return paragraphs


def closest_color(color, highlight_colors):
    # Compute euclidean distance to each known color

    if color[0] == -2: # Special case for StrikeOut
        return "StrikeOut"
    
    best_name = None
    best_dist = float('inf')
    for known_rgb, name in highlight_colors.items():
        # Ensure both tuples are same length
        dist = math.sqrt(sum((c - k) ** 2 for c, k in zip(color, known_rgb)))
        if dist < best_dist:
            best_dist = dist
            best_name = name

    return best_name

def get_parser_logger(
    log_file: Union[str, bool, None] = "parse_pdf.log",
    level: Union[int, str] = logging.DEBUG,
) -> logging.Logger:
    logger = logging.getLogger("pdf_parser")
    for h in logger.handlers[:]:
        logger.removeHandler(h)
        
    if isinstance(level, str):
        level = getattr(logging, level.upper(), logging.DEBUG)
    logger.setLevel(level)
    fmt = logging.Formatter("%(levelname)s: %(message)s")

    # No handlers at all
    if log_file is None:
        return logger

    # Console only
    if log_file is False:
        ch = logging.StreamHandler()
        ch.setFormatter(fmt)
        logger.addHandler(ch)
        return logger

    # File only
    fh = logging.FileHandler(log_file, mode="w", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

def text_to_json(paragraphs,
                        note_symbol     =   ['<<', '>>'],
                        highlight_symbol=   ['{', '}'],
                        comment_symbol  =   ['[', ']'],
                        annot_conf_symbol=  ['/!{', '/!}'],
                       context_size=20,
                       logger: Optional[logging.Logger] = None):
    result = {'paragraphs': []}

    for idx, para in enumerate(paragraphs):
        logger.debug(f"text_to_json: index {idx} / {len(paragraphs)} processing paragraph: {para[:50]}...")
        cleaned_parts = []
        annotations = []
        i = 0
        N = len(para)
        # First pass: extract annotations, build cleaned_parts, record spans
        logger.debug(f"text_to_json: First pass: Extract annotations")
        while i < N:
            logger.debug(f"text_to_json: Processing char {i}/{N}: {para[i]} in paragraph {idx}")
            if para.startswith(highlight_symbol[0], i):
                # highlight text
                start_text = i + len(highlight_symbol[0])
                end_text = para.find(highlight_symbol[1], start_text)
                text = para[start_text:end_text]
                # color
                start_col = para.find(annot_conf_symbol[0], end_text) + len(annot_conf_symbol[0])
                end_col = para.find(annot_conf_symbol[1], start_col)
                color = para[start_col:end_col]
                # comment
                start_com = para.find(comment_symbol[0], end_col) + len(comment_symbol[0])
                end_com = para.find(comment_symbol[1], start_com)
                comment = para[start_com:end_com]

                start_idx = sum(len(p) for p in cleaned_parts)
                cleaned_parts.append(text)
                end_idx = start_idx + len(text)

                annotations.append({
                    'type': 'highlight',
                    'text': text,
                    'color': color,
                    'comment': comment,
                    'span': (start_idx, end_idx)
                })

                i = end_com + len(comment_symbol[1])

            elif para.startswith(note_symbol[0], i):
                # note comment
                start_note = i + len(note_symbol[0])
                end_note = para.find(note_symbol[1], start_note)
                comment = para[start_note:end_note]

                start_idx = sum(len(p) for p in cleaned_parts)
                # notes don't insert body text
                end_idx = start_idx

                annotations.append({
                    'type': 'note',
                    'comment': comment,
                    'span': (start_idx, end_idx)
                })

                i = end_note + len(note_symbol[1])

            else:
                # regular char
                # logger.debug(f"text_to_json: Regular char at {i}: {para[i]}")
                cleaned_parts.append(para[i])
                i += 1

        body = ''.join(cleaned_parts).strip()

        # Second pass: compute context_left/right using spans
        logger.debug(f"text_to_json: Second pass: Compute context")
        for ann in annotations:
            start_idx, end_idx = ann['span']
            ann['context_left'] = body[max(0, start_idx - context_size):start_idx]
            ann['context_right'] = body[end_idx:end_idx + context_size]

            # remove span key—no longer needed
            del ann['span']

        result['paragraphs'].append({
            'body': body,
            'annotations': annotations
        })

    return result

def char_type(txt,
              highlight_symbol,
              comment_symbol,
              note_symbol,
              annot_conf_symbol):
    """
    Determines the type of character/string at the end of txt.

    Returns:
        "highlight-open" or "highlight-close" for highlight
        "comment-open" or "comment-close" for comment
        "note-open" or "note-close" for note
        "annot_conf-open" or "annot_conf-close" for annotation config
        "sentence-end" when certain the sentence ended
        "char" for alphabetical characters and dashes
        "unknown" for anything else
    """
    # Strip ALL trailing spaces/newlines so they don't affect our checks
    txt = txt.rstrip(' \n')
    if not txt:
        return "unknown"

    # Check symbol pairs
    symbol_sets = {
        'highlight': highlight_symbol,
        'comment':    comment_symbol,
        'note':       note_symbol,
        'annot_conf': annot_conf_symbol,
    }
    for name, (open_sym, close_sym) in symbol_sets.items():
        if txt.endswith(open_sym):
            return f"{name}-open"
        if txt.endswith(close_sym):
            return f"{name}-close"

    # Sentence-end check: '.', '!', or '?'
    last = txt[-1]
    if last in {'.', '!', '?'}:
        # If it's a dot, avoid abbreviations: require that two chars back is NOT also a dot
        if last == '.':
            if len(txt) >= 3 and txt[-3] == '.' and txt[2].isalpha():
                # part of "..." or "U.S."
                return "char"
            else:
                return "sentence-end"
        else:
            # '!' or '?' always mean end of sentence here
            return "sentence-end"

    # Plain character: letter or dash
    if last.isalpha() or last in {'-', ',', ';', ':', '('}:
        return "char"

    return "unknown"

def combine_paragraphs(
    paragraphs: List[str],
    section_list: List[str],
    highlight_symbol: tuple,
    comment_symbol: tuple,
    note_symbol: tuple,
    annot_conf_symbol: tuple,
    logger: Optional[logging.Logger] = None
) -> List[str]:
    """
    Combines paragraphs that were broken up by the newpage.
    Handles broken highlights, and also merges across trailing comments or notes
    when the underlying text should be treated as continuing.
    """
    if logger is None:
        logger = logging.getLogger(__name__)
        logger.addHandler(logging.NullHandler())

    # Pre-unpack symbols
    hc_open, hc_close = highlight_symbol
    co_open, co_close = comment_symbol
    nc_open, nc_close = note_symbol
    ac_open, ac_close = annot_conf_symbol

    # maximum context length to look at the end of the previous para
    max_len = max(
        len(s) for s in (
            highlight_symbol +
            comment_symbol +
            note_symbol +
            annot_conf_symbol
        )
    ) * 2

    i = 1
    while i < len(paragraphs):
        prev = paragraphs[i-1]
        curr = paragraphs[i]
        prev_tail = prev.rstrip()[-max_len:]
        curr_stripped = curr.lstrip()

        # SPECIAL: broken highlight with empty comment on prev, continued on curr
        if prev_tail.endswith(co_close) and curr_stripped.startswith(hc_open):
            try:
                # locate empty comment immediately after an annot_conf close
                co_open_idx = prev.rfind(co_open)
                if co_open_idx < 0 or prev[co_open_idx + len(co_open) : prev.rfind(co_close)] != "":
                    raise ValueError("Comment not actually empty")

                ac_close_idx = prev.rfind(ac_close, 0, co_open_idx)
                if ac_close_idx < 0:
                    raise ValueError("Missing annot_conf close before empty comment")

                ac_open_idx = prev.rfind(ac_open, 0, ac_close_idx)
                if ac_open_idx < 0:
                    raise ValueError("Missing annot_conf open before its close")

            except ValueError as e:
                logger.warning("Special merge check failed: %s (prev_tail=%r)", e, prev_tail)
            else:
                h_close_idx_prev = prev.rfind(hc_close, 0, ac_open_idx)
                if h_close_idx_prev < 0:
                    logger.warning(
                        "Could not find highlight-close prior to annot_conf on prev; skipping special merge."
                    )
                else:
                    prefix = prev[:h_close_idx_prev]
                    body_start = len(hc_open)
                    body_end = curr_stripped.find(hc_close, body_start)
                    if body_end < 0:
                        logger.warning(
                            "Could not find highlight-close on curr despite it starting with open; skipping special merge."
                        )
                    else:
                        body = curr_stripped[body_start:body_end]
                        meta = curr_stripped[body_end:]
                        sep = "" if prev.endswith(" ") or curr.startswith(" ") else " "
                        new_para = prefix + sep + body + meta

                        logger.debug(
                            "Special broken-highlight merge: prev_tail=%r curr_head=%r -> %r",
                            prev_tail, curr_stripped[:len(hc_open)+10], new_para[-50:]
                        )
                        paragraphs[i-1] = new_para
                        del paragraphs[i]
                        continue  # re-check at same index

        # LOOK-THROUGH COMMENT -> ANNOT_CONF -> HIGHLIGHT_CLOSE
        base_prev = None
        if prev_tail.endswith(co_close):
            co_open_idx = prev.rfind(co_open)
            if co_open_idx >= 0:
                temp = prev[:co_open_idx]
                # strip trailing annot_conf
                if temp.endswith(ac_close):
                    ac_open_idx = temp.rfind(ac_open)
                    if ac_open_idx >= 0:
                        temp = temp[:ac_open_idx]
                # strip trailing highlight-close
                if temp.endswith(hc_close):
                    temp = temp[: -len(hc_close)]
                base_prev = temp
                logger.debug(
                    "Peeling comment+config+highlight for logic: %r",
                    base_prev.rstrip()[-max_len:]
                )

        # LOOK-THROUGH NOTE
        if base_prev is None and prev_tail.endswith(nc_close):
            nc_open_idx = prev.rfind(nc_open)
            if nc_open_idx >= 0:
                base_prev = prev[:nc_open_idx]
                logger.debug(
                    "Peeling note for merge logic: %r",
                    base_prev.rstrip()[-max_len:]
                )

        # Determine the logic tail
        if base_prev is not None:
            logic_tail = base_prev.rstrip()[-max_len:]
        else:
            logic_tail = prev.rstrip()[-max_len:]

        # SIMPLE MERGE or KEEP SEPARATE
        from_char = char_type(
            logic_tail,
            highlight_symbol,
            comment_symbol,
            note_symbol,
            annot_conf_symbol
        )
        first_char = curr_stripped[0] if curr_stripped else ""

        # SIMPLE MERGE: ends with char + next starts lowercase
        if from_char == "char" and (not prev in section_list):# and first_char.islower():
            sep = "" if prev.endswith(" ") or curr_stripped.startswith(" ") else " "
            logger.debug(
                "Merging paragraphs because logic_tail ends with char (%r) and curr starts lowercase (%r). UPDATE: Lowercase requirement is dismissed.",
                logic_tail[-1], first_char
            )
            paragraphs[i-1] = prev + sep + curr_stripped
            del paragraphs[i]
            continue

        # KEEP SEPARATE: ends sentence + next starts uppercase
        if from_char == "sentence-end" and first_char.isupper():
            logger.debug(
                "Keeping as two paras: logic_tail ends sentence (%r), curr starts uppercase (%r).",
                logic_tail[-1], first_char
            )
            i += 1
            continue

        # ALL OTHER CASES: no merge
        logger.debug(
            "No merge: logic_tail=%r (type=%s), curr_head=%r",
            logic_tail, from_char, first_char
        )
        i += 1

    return paragraphs

def block_type(page_no,
                block_cnt,
                block,
                page,
                logger: Optional[logging.Logger] = None):
    """
    Determines if the block is a date or name
    
    Returns:
        "date"      if is a date
        "name"      if is a name
        "title"     if is a title
        "section"   if is a section
        "text"      if is a text
        "unknown"   if is unknown
    """
    
    x0, y0, x1, y1, block_text = block[:5]

    if len(block_text) < 2:
        logger.debug(f"block_type: Block text too short, remove junk")
        return "name"

    # Title
    #logger.debug(f"Is it title? {page_no == 0} {len(block_text) <= 50} {(x1-x0) < 0.5 * (page.mediabox.x1-page.mediabox.x0)} {((x0-page.mediabox.x0)-(page.mediabox.x1-x1))}")
    #logger.debug(f"x0: {x0} page.mediabox.x0: {page.mediabox.x0} x1: {x1} page.mediabox.x1: {page.mediabox.x1} (x0-page.mediabox.x0): {(x0-page.mediabox.x0)} (page.mediabox.x1-x1): {(page.mediabox.x1-x1)}")
    logger.warning(f"Is it title? len(block_text) <= 50: {len(block_text) <= 50} abs((x0-page.mediabox.x0)-(page.mediabox.x1-x1)) < 30) x0:{x0} x1: {x1} page.mediabox.x0:{page.mediabox.x0} page.mediabox.x1:{page.mediabox.x1}")
    if (len(block_text) <= 50
        and (x1-x0) < 0.5 * (page.mediabox.x1-page.mediabox.x0)
        and abs((x0-page.mediabox.x0)-(page.mediabox.x1-x1)) < 30):
        logger.debug(f"block_type: It's in the middle of the page, title or section.")
        if page_no == 0:
            return "title"
        else:
            return "section"
    

    # Date
    DATE_PATTERNS = [
        # numeric: 01/02/2024 or 1/2/24 or 10.02.25
        r"\d{1,2}[\/\.]\d{1,2}[\/\.]\d{2,4}",
        # day + month name + year: 10 Dec 2025 or 22 November 24
        r"\d{1,2}\s+(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|"
        r"Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|"
        r"Nov(?:ember)?|Dec(?:ember)?)\s+\d{2,4}",
        # month name + year: Nov 2023 or july 2025
        r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|"
        r"Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|"
        r"Nov(?:ember)?|Dec(?:ember)?)\s+\d{2,4}",
    ]
    text = block_text.strip()
    for pattern in DATE_PATTERNS:
        #if re.fullmatch(pattern, text, flags=re.IGNORECASE):
        if re.search(pattern, text, flags=re.IGNORECASE) and len(block_text) < 25:
            logger.debug(f"block_type: Matched date pattern")
            return "date"
    
    # Junk (Top right). Pass it as a name. Must be removed
    if (page.mediabox.x1 - x1) < 0.5 * (x0-page.mediabox.x0) and \
        x0 > page.mediabox.x0+(page.mediabox.x1 - page.mediabox.x0)/2:
        logger.debug(f"block_type: Top right junk.")
        return "name"
    
    # On the left of the page
    if (x0-page.mediabox.x0) < 0.5 * (page.mediabox.x1 - x1) and \
        ':' in block_text[-2:]: # Most definitely a section
        logger.debug(f"block_type: Left of the page, and ending with a colon. ")
        return "section"
    
    if (x1-x0)/(y1-y0) < 1:
        logger.warning(f"text too heighty and less widthy! Unknown")
        return "unknown"

    #logger.debug(f"square like? (x1-x0): {(x1-x0)} (y1-y0): {(y1-y0)} {(x1-x0)/(y1-y0) < 2} {x1 < (page.mediabox.x0+page.mediabox.x1)/2} {len(block_text) < 100}")
    if (x1-x0)/(y1-y0) < 3 and \
        x1 < (page.mediabox.x0+page.mediabox.x1)/2 and \
            len(block_text) < 100: # Too Square like
        logger.warning("block_type: Too square like. Passing it as a name so it would be removed.")
        return "name"
    
    if len(block_text) > 45:
        logger.debug(f"block_type: Too long text. It's text.")
        return "text"
        
    
    clean = block_text.replace('\n', '').replace(' ', '')
    if not clean:
        logger.warning("block_type: Empty block! Passing it as a name so it would be removed.")
        return "name" # Empty block...?
    if (len(clean) <= 3 \
        and clean.isdigit()):
        logger.debug(f"block_type: It's a page number")
        return "name" # It's the page number. Passing as name so it would be removed.

    logger.warning("block_type: Unknown block type")
    return "unknown"
    
def similar(a: str, b: str, threshold: float = 0.8) -> bool:
    """
    Checks if two strings are similar enough based on a threshold.
    """
    if not a and not b:
        return True
    
    if len(a) < 3 or len(b) < 3:
        return True

    if not a or not b:
        return False
    
    a_norm = "".join(a.split()).lower()
    b_norm = "".join(b.split()).lower()
    ratio = SequenceMatcher(None, a_norm, b_norm).ratio()
    return ratio >= threshold


def combine_to_json(
    paragraphs: List[str],
    highlights: List[Dict],
    sticky_notes: List[Dict],
    context_window: int = 20,
    logger: Optional[logging.Logger] = None
) -> Dict:
    """
    Combines paragraphs, highlights, and sticky notes into a structured JSON format.
    """
    if logger is None:
        logger = logging.getLogger(__name__)
        logger.addHandler(logging.NullHandler())

    def normalize(s: str) -> str:
        """Remove all whitespace for matching."""
        return re.sub(r"\s+", "", s or "")

    def build_index_map(text: str) -> List[int]:
        """Map indices in normalized text back to original text indices."""
        mapping = []
        for i, ch in enumerate(text):
            if not ch.isspace():
                mapping.append(i)
        return mapping

    def adjust_boundary(start: int, end: int, para: str, prov_cl: str, prov_cr: str):
        """
        Adjusts highlight start/end to include or exclude partial words based on prov_cl and prov_cr.
        Left: include full word if highlighted head covers >=50%, else drop head fragment.
        Right: include full word if tail covers >=50%, else drop tail fragment.
        """
        # Determine full word around start
        left_ws = para.rfind(' ', 0, start) + 1
        right_ws = para.find(' ', start)
        if right_ws < 0:
            right_ws = len(para)
        full_word = para[left_ws:right_ws]
        # head fragment from highlight boundary
        head = ''
        for i in range(start, end):
            if para[i].isspace():
                break
            head += para[i]
        # adjust left
        if full_word and head:
            if len(head) >= 0.5 * len(full_word):
                new_start = left_ws
            else:
                new_start = start + len(head)
        else:
            new_start = start

        # Determine full word around end
        left_ws_r = para.rfind(' ', 0, end) + 1
        right_ws_r = para.find(' ', end)
        if right_ws_r < 0:
            right_ws_r = len(para)
        full_word_r = para[left_ws_r:right_ws_r]
        # tail fragment before end
        tail = ''
        for i in range(end - 1, -1, -1):
            if para[i].isspace():
                break
            tail = para[i] + tail
        # adjust right
        if full_word_r and tail:
            if len(tail) >= 0.5 * len(full_word_r):
                new_end = right_ws_r
            else:
                new_end = end - len(tail)
        else:
            new_end = end

        return new_start, new_end

    output = {"paragraphs": []}
    per_par_annotations = [[] for _ in paragraphs]
    unmatched = []

    # Prepare unified annotations
    annotations = []
    for hl in highlights:
        annotations.append({
            "type": "highlight",
            "orig_text": hl.get("h_text", "").replace("\n", " "),
            "comment": hl.get("content", ""),
            "ctx_left_in": hl.get("context_left", ""),
            "ctx_right_in": hl.get("context_right", ""),
            "color": hl.get("color", ""),
            "used": False
        })
    for note in sticky_notes:
        annotations.append({
            "type": "note",
            "orig_line": note.get("line", ""),
            "orig_text": note.get("local-text", "").replace("\n", " "),
            "comment": note.get("comment", ""),
            "ctx_left_in": note.get("context_left", "").replace("\n", " "),
            "ctx_right_in": note.get("context_right", "").replace("\n", " "),
            "used": False
        })

    # Match
    for ann in annotations:
        matched = False
        for p_idx, para in enumerate(paragraphs):
            norm_para = normalize(para)
            idx_map = build_index_map(para)

            if ann["type"] == "highlight":
                prov_cl_norm = normalize(ann.get("ctx_left_in", ""))
                prov_cr_norm = normalize(ann.get("ctx_right_in", ""))
                tgt_norm = normalize(ann["orig_text"])
                pos_norm = norm_para.find(tgt_norm)
                if pos_norm < 0:
                    continue
                # Check if independent contexts exist
                if (not prov_cl_norm or prov_cl_norm in norm_para) and (not prov_cr_norm or prov_cr_norm in norm_para):
                    # Found all parts independently
                    start0 = idx_map[pos_norm]
                    end0 = idx_map[pos_norm + len(tgt_norm) - 1] + 1
                else:
                    # Strict match combinations
                    combos = []
                    if prov_cl_norm and prov_cr_norm:
                        combos = [prov_cl_norm + tgt_norm + prov_cr_norm,
                                  prov_cl_norm[:-1] + tgt_norm + prov_cr_norm,
                                  prov_cl_norm + tgt_norm + prov_cr_norm[1:],
                                  prov_cl_norm[:-1] + tgt_norm + prov_cr_norm[1:]]
                    elif prov_cl_norm:
                        combos = [prov_cl_norm + tgt_norm, prov_cl_norm[:-1] + tgt_norm]
                    elif prov_cr_norm:
                        combos = [tgt_norm + prov_cr_norm, tgt_norm + prov_cr_norm[1:]]
                    else:
                        combos = [tgt_norm]
                    found_combo = False
                    for combo in combos:
                        combo_pos = norm_para.find(combo)
                        if combo_pos >= 0:
                            # h_text segment within combo
                            inner_pos = combo.find(tgt_norm)
                            pos_norm = combo_pos + inner_pos
                            start0 = idx_map[pos_norm]
                            end0 = idx_map[pos_norm + len(tgt_norm) - 1] + 1
                            found_combo = True
                            break
                    if not found_combo:
                        continue
                # Adjust boundaries
                # Only for actual highlight colors, not for StrikeOuts
                if ann['color'] == "StrikeOut":
                    new_start, new_end = start0, end0
                    
                else:
                    new_start, new_end = adjust_boundary(start0, end0, para, ann["ctx_left_in"], ann["ctx_right_in"])
                # Extract normalized adjusted span and recalc positions
                adjusted_norm = normalize(para[new_start:new_end])
                new_pos_norm = norm_para.find(adjusted_norm)
                if new_pos_norm < 0:
                    continue
                new_end_norm = new_pos_norm + len(adjusted_norm)
                # Finally extract context surroundings
                start, end = new_start, new_end
                cl = para[max(0, start - context_window):start]
                cr = para[end:end + context_window]
                text = para[start:end]
                entry = {"type": "highlight", "color": ann["color"], "text": text,
                         "comment": ann["comment"], "context_left": cl, "context_right": cr}
                # StrikeOuts can be treated as highlights till here, but now should be saved as StrikeOuts
                if ann['color'] == "StrikeOut":
                    entry = {"type": "strikeout", "text": text,
                         "comment": ann["comment"], "context_left": cl, "context_right": cr}


            else:
                # Note matching
                prov_cl = ann.get("ctx_left_in", "")
                prov_cr = ann.get("ctx_right_in", "")
                tgt_line_norm = normalize(ann["orig_line"])
                pos_norm = normalize(para).find(tgt_line_norm)
                if pos_norm < 0:
                    continue
                idx_map = build_index_map(para)
                ls = idx_map[pos_norm]
                le = idx_map[pos_norm + len(tgt_line_norm) - 1] + 1
                local_norm = normalize(ann.get("orig_text", ""))
                if local_norm:
                    sub = normalize(para[ls:le])
                    sub_pos = sub.find(local_norm)
                    if sub_pos >= 0:
                        start = idx_map[pos_norm + sub_pos]
                        end = idx_map[pos_norm + sub_pos + len(local_norm) - 1] + 1
                    else:
                        start = le
                        end = le
                else:
                    start = le
                    end = le
                # Extend contexts only if provided
                if prov_cl:
                    if len(prov_cl) < context_window:
                        extra_len = context_window - len(prov_cl)
                        extra_start = max(0, start - extra_len)
                        extra = para[extra_start:start]
                        new_cl = extra + prov_cl
                    else:
                        new_cl = prov_cl
                else:
                    new_cl = ""
                if prov_cr:
                    if len(prov_cr) < context_window:
                        extra_len = context_window - len(prov_cr)
                        extra = para[end:end + extra_len]
                        new_cr = prov_cr + extra
                    else:
                        new_cr = prov_cr
                else:
                    new_cr = ""
                entry = {"type": "note", "comment": ann["comment"], "context_left": new_cl, "context_right": new_cr}

            per_par_annotations[p_idx].append(entry)
            ann["used"] = True
            matched = True
            break
        if not matched:
            logger.error("NOT fatal: unmatched annotation: %s", ann)
            unmatched.append(ann)

    # Build output paragraphs
    for i, p in enumerate(paragraphs, 1):
        output["paragraphs"].append({"body": p,
                                      "annotations": per_par_annotations[i - 1]})
    if unmatched:
        logger.error("NOT fatal: unmatched annotations present")
        extra = []
        for ann in unmatched:
            if ann["type"] == "highlight":
                extra.append({"type": "highlight", "color": ann.get("color", ""),
                              "text": ann.get("orig_text", ""), "comment": ann.get("comment", ""),
                              "context_left": ann.get("ctx_left_in", ""),
                              "context_right": ann.get("ctx_right_in", "")})
            else:
                extra.append({"type": "note", "comment": ann.get("comment", ""),
                              "context_left": ann.get("ctx_left_in", ""),
                              "context_right": ann.get("ctx_right_in", "")})
        output["paragraphs"].append({"body": "", "annotations": extra})
    return output


# Parse PDF V2.1

In [ ]:
def parse_pdf(
        pdf_path:   str,
        note_symbol     =   ['<<', '>>'],
        highlight_symbol=   ['{', '}'],
        comment_symbol  =   ['[', ']'],
        annot_conf_symbol=  ['/!{', '/!}'],
        highlight_colors=   {(254.89, 185.05, 241.04):'pink',
                             (153.04, 234.89, 163.89):'green',
                             (252.06, 229.93, 127.97):'yellow'},
        new_parag_symbol=   "$#NEWPARAGRAPH#$",
        new_page_symbol =   "$#NEWPAGE#$",
        log_file:   str =   "parse_pdf.log" # False: Console, None: No log
        ) -> str:
    """
    Open PDF, iterate each page, and:
      + Pull text in layout-ordered blocks.
      + Collect all highlight and Text (sticky-note) annotations with their rectangles.
      + For each block, if it intersects a highlight rect and the exact highlighted text is found,
        replace that text with: original + [comment].
      + If it intersects a sticky-note rect, append [comment] at block end.
    Returns a single string of the whole document.
    """
    logger = get_parser_logger(log_file)
    logger.info(f"Opening {pdf_path!r}")
    # x_margin, y_margin = 1, 15

    doc = fitz.open(pdf_path)
    output_blocks = []
    date_name_block_ind = [] # Indices of blocks that are date and name
    page_paragraphs = [] # List of lists. For each page, there is a list of paragraphs

    page_no = -1
    first_text = False # Have we seen the first text of the page? If not, remove the "unknown" blocks
    block_types = [] # List of block types. This is so that the text after the title would go in a different paragraph
    section_list = [] # List of sections and the titles so its paragraph wouldn't be merged with the next text
    
    highlights = []  # List of Dict: h_text, content, context_left, context_right, color
    notes      = []  # list of (Rect, comment, used). INTERMEDIATE, sticky_notes is used in the output
    sticky_notes=[]  # List of Dict: line, local-text, comment, context_left, context_right, used
    
    for page in doc:
        page_paragraphs.append([])
        page_no += 1
        logger.info(f">>>>>>> New Page!")
        output_blocks.append(new_page_symbol)
        # grab and sort text blocks top->bottom, left->right
        blocks = page.get_text("blocks")  # each: x0, y0, x1, y1, "text", block_no
        leftmost, rightmost, line_height = 1e20, -1, 1e10
        for x0, y0, x1, y1, block_text, block_no1, block_no2 in blocks:
            logger.debug(f"TEXT: ({x0}, {y0}, {x1}, {y1}, \"{block_text[0:10]}...\", {block_no1}, {block_no2}),")
            leftmost    =  min(leftmost, x0)
            rightmost   =  max(rightmost, x1)
            if len(block_text) > 20: # Considerable number of characters. It's a normal line
                logger.debug("Normal line of text. Adjust line_height")
                break_line_cnt = block_text.count('\n')
                logger.debug(f"break_line_cnt: {break_line_cnt}")
                if break_line_cnt > 1:
                    line_height = min(line_height, (y1 - y0) / break_line_cnt)
                    logger.debug(f"Block has more than one line, line height: {line_height}")
                else:
                    line_height = min(line_height, (y1 - y0))
                    logger.debug(f"Block has only one line, line height: {line_height}")
        logger.info(f"Leftmost: {leftmost}, Rightmost: {rightmost}, Line height: {line_height}")

        if not blocks:
            logger.warning("Page with no blocks... Skipped")
            continue
        
        blocks.sort(key=lambda b: (b[1], b[0]))

        # Collect annotations
        annotations = page.annots()
        if annotations:
            for annot in annotations:
                #print("\nANNOT:", annot)
                subtype = annot.type[1]  # "Highlight", "Text"
                rect    = annot.rect     # bounding box of the annotation
                content = annot.info.get("content", "").strip()

                h_quads = [] 
                logger.debug(f"Annotation: {subtype}, {rect}, {content}")

                if subtype == "Highlight" or subtype == "StrikeOut":
                    if subtype == "Highlight":
                        color   = annot.colors.get("stroke") or annot.colors.get("fill") 
                        pix = page.get_pixmap(clip=fitz.Quad(annot.vertices[0:4]).rect, dpi=150)
                        arr = np.frombuffer(pix.samples, dtype=np.uint8)
                        arr = arr.reshape(pix.height, pix.width, pix.n)
                        avg_color = arr.mean(axis=(0, 1))  # e.g. [R, G, B]
                        logger.debug(f"Average RGB:{avg_color} {closest_color(avg_color, highlight_colors)}")
                    if subtype == "StrikeOut":
                        logger.debug("Strikeout annotation found.")
                        avg_color = (-2, -2, -2) # Special color for strikeout

                    # Get exact highlighted text via annotation quads
                    exact = []
                    # annot.vertices holds points in groups of 4 for each quad
                    if len(annot.vertices) < 4:
                        logger.warning("Warning: Highlight annotation has less than 4 vertices:", annot.vertices)
                        continue

                    # 3 Blocks: first, middle, last. Each handled separately.
                    first_quad = fitz.Quad(annot.vertices[0:4])                    
                    last_quad  = fitz.Quad(annot.vertices[-4:])

                    h_text = page.get_text("text", clip=rect).strip()
                    # Remove top left and bottom right
                    topleft_rect = fitz.Rect(page.mediabox.x0, first_quad.rect.y0, first_quad.rect.x0, first_quad.rect.y1)
                    bottomright_rect = fitz.Rect(last_quad.rect.x1, last_quad.rect.y0, page.mediabox.x1, last_quad.rect.y1)
                    topleft_txt = page.get_text("text", clip=topleft_rect).strip()
                    bottomright_txt = page.get_text("text", clip=bottomright_rect).strip()
                    logger.debug(f"Whole highlight rect:{h_text}\nTop left txt: {topleft_txt}\nBottom right txt:{bottomright_txt}")


                    if not h_text:
                        logger.warning("HIGHLIGHT RECT EMPTY!")
                        pass

                    if not topleft_txt or \
                        similar(topleft_txt, h_text[:len(topleft_txt)]):
                        logger.debug("Safely removing top left text")
                        if topleft_txt:
                            h_text = h_text[len(topleft_txt):].strip()
                        logger.debug(f"highlight after removal: {h_text}")
                    else:
                        logger.warning(f"Top left text|{topleft_txt}| is not similar to the highlighted text |{h_text[:len(topleft_txt)]}|. Not removing it.")
                    
                    if not h_text:
                        logger.warning("HIGHLIGHT RECT EMPTY!")
                        pass

                    if not bottomright_txt or \
                        similar(bottomright_txt, h_text[-len(bottomright_txt):]):
                        logger.debug("Safely removing bottom right text")
                        if bottomright_txt:
                            h_text = h_text[:-len(bottomright_txt)].strip()
                        logger.debug(f"highlight after removal: {h_text}")
                    else:
                        logger.warning(f"Bottom right text {bottomright_txt} is not similar to the highlighted text {h_text[-len(bottomright_txt):]}. Not removing it.")

                    if not h_text:
                        logger.warning("HIGHLIGHT RECT EMPTY!")
                        pass
                    
                    logger.debug(f"highlight after removal: {h_text}")
                    if h_text:
                        highlights.append({'h_text':h_text, 'content': content, 'context_left': topleft_txt, 'context_right': bottomright_txt, 'color': closest_color(avg_color, highlight_colors)})
                        logger.debug(f"Highlighted text (not used): {h_text} Quads (used): {h_quads} content is: {content}")

                elif subtype == "Text":
                    notes.append([rect, content, False]) 
                else:
                    logger.warning(f"Different annotation type: {subtype} with rect: {rect} and content: {content}")
                    notes.append([rect, content, False])

        # Process each block
        # Sort highlights by the y0 value, then by x0
        # highlights.sort(key=lambda b: (b[1][0].rect.y0, b[1][0].rect.x0))

        """
        logger.info(f"Combine the sticky note that coincide with highlights")
        for i, (rect, h_quads, content, color) in enumerate(highlights):
            for j, (note_rect, note_text, used) in enumerate(notes):
                if used:
                    continue
                if rect.y0 <= note_rect.y0 and rect.y1 >= note_rect.y1: 
                    logger.info(f"Sticky note intersects with highlight: {content} {note_text}")
                    highlights[i] = (rect, h_quads, content + " " + note_text, color)
                    notes[j][2] = True  # Mark this note as used
        """
                    

        buffer_rect = fitz.Rect(0, 0, 0, 0) # Buffer Must be parsed
        note_buffer = []
        bottomright_buffer = None
        # If buffer has no intersection with the next thing we are trying to parse, then just write it
        # Otherwise, break it up into the correct pieces and handle it

        min_block_distance = 1e10
        for i in range(1, len(blocks)):
            min_block_distance = min(min_block_distance, blocks[i][1]-blocks[i-1][3])
        logger.info(f"min_block_distance: {min_block_distance}")
        logger.info(f"Process each block. Buffer rect: {buffer_rect}")

        block_cnt = -1 # Counting the blocks, for adding very top and very bottom sticky notes
        first_text = False # Have we seen the first text of the page? If not, remove the "unknown" blocks
        marked_mid_quads = [False for _ in range(len(highlights))] # List of indices of highlights that mid quad is placed. If the highlight spans different blocks, the middle quad is going to be inserted for each one of the blocks. For PDFs were each line is a block
        for block in blocks:
            #logger.info(f"BLOCK LIST {output_blocks}")
            new_paragraph_buffer = ""
        #for block in [[0, 0, page.mediabox.x1, page.mediabox.y1, ""]]:
            block_cnt += 1

            logger.info(f"New block {block_cnt}")
            if len(block) < 5:
                logger.warning(f" Block len < 5: {block}")
                continue

            x0, y0, x1, y1, block_text = block[:5]
            logger.info(f"Preview text: {block_text}...")
            must_remove_block = block_type(page_no=page_no, block_cnt=block_cnt, block=block, page=page, logger=logger)
            logger.info(f"block_type? {must_remove_block}")
            block_types.append(must_remove_block)
            if must_remove_block == "title" or must_remove_block == "section":
                if block_text.strip():
                    section_list.append(block_text.strip())
            if must_remove_block == "unknown":
                logger.info(f"Unknown block will be removed: {not first_text}")
            if must_remove_block == "text" or must_remove_block == "title": 
                first_text = True # Keep the "Unknowns"
            if (must_remove_block == 'name' or must_remove_block == 'date' or must_remove_block == 'unknown') and not first_text:
                date_name_block_ind.append(len(output_blocks))
            
            b_rect = fitz.Rect(x0, y0, x1, y1)

            logger.debug("Checking if this block is start of a new paragraph")
 
            logger.debug(f"indented? {block_cnt > 1} and {b_rect.x0 > leftmost} and {b_rect.x0 < (rightmost-leftmost)/2} and {blocks[block_cnt-1][0] < b_rect.x0} and {abs(blocks[block_cnt-1][0] - leftmost) <= 5*line_height}") 
          
            logger.debug(f"larger distance? {block_cnt > 1} and {b_rect.x0-blocks[block_cnt-1][3] > min_block_distance} and {((blocks[block_cnt-1][0]-blocks[block_cnt-2][3]) < (b_rect.x0-blocks[block_cnt-1][3])) if (block_cnt > 2) else True}")
            logger.debug(f"len(blocks: {len(blocks)} < 12")

            if len(block_types) > 1 and block_types[-2] == "title" and \
                block_types[-1] != "title":
                logger.info("Previous block was a title. This is a new paragraph")
                new_paragraph_buffer = new_parag_symbol
            
            logger.debug(f"len(blocks) {len(blocks)} (blocks[-1][3]-blocks[0][1]) {(blocks[-1][3]-blocks[0][1])} (page.mediabox.y1-page.mediabox.y0) {(page.mediabox.y1-page.mediabox.y0)} devision: {(blocks[-1][3]-blocks[0][1])/(page.mediabox.y1-page.mediabox.y0)} {len(blocks)/(blocks[-1][3]-blocks[0][1])*(page.mediabox.y1-page.mediabox.y0)+1}")
            if len(blocks)/(blocks[-1][3]-blocks[0][1])*(page.mediabox.y1-page.mediabox.y0)+1 < 18:
                logger.info("new paragraph. Probably every block is a new paragraph")
                new_paragraph_buffer = new_parag_symbol
                first_newline = block_text.find('\n')
                if first_newline != -1:
                    preview = block_text[:first_newline]
                else:
                    preview = block_text[:30]
                page_paragraphs[-1].append(preview)
            
            elif (
                block_cnt > 1
                and b_rect.x0 > leftmost
                and b_rect.x0 < (rightmost+leftmost)/2
                and blocks[block_cnt-1][0] < b_rect.x0
                and abs(blocks[block_cnt-1][0] - leftmost) <= 5*line_height
            ):
                new_paragraph_buffer = new_parag_symbol
                logger.info("new paragraph, because of starting with a larger x0")
                first_newline = block_text.find('\n')
                if first_newline != -1:
                    preview = block_text[:first_newline]
                else:
                    preview = block_text[:30]
                page_paragraphs[-1].append(preview)
            
            elif (
                block_cnt > 1
                and b_rect.y0-blocks[block_cnt-1][3] > min_block_distance*1.5
                and (
                    ((blocks[block_cnt-1][1]-blocks[block_cnt-2][3])*1.3 < (b_rect.y0-blocks[block_cnt-1][3]))
                    if (block_cnt > 2)
                    else True
                )
            ):
                new_paragraph_buffer = new_parag_symbol
                logger.info("new paragraph, because distance is larger to the previous blocks")
                first_newline = block_text.find('\n')
                if first_newline != -1:
                    preview = block_text[:first_newline]
                else:
                    preview = block_text[:30]
                page_paragraphs[-1].append(preview)
            

            pattern = r"\n *\n"
            matches = list(re.finditer(pattern, block_text))
            positions = [(m.start(), m.end()) for m in matches]
            ## Prevous method
            for st, end in positions:
                logger.info(f"new paragraph inside block. preview: {block_text[end:end+30]}")
                first_newline = block_text[end:].find('\n')
                if first_newline != -1:
                    preview = block_text[end:end+first_newline]
                else:
                    preview = block_text[end:end+30]
                page_paragraphs[-1].append(preview)
            ## New method
            for st, end in sorted(positions, key=lambda x: x[1], reverse=True):
                block_text = block_text[:end] + new_parag_symbol + block_text[end:]
                logger.info(f"inserted the new paragraph symbol within the block: {block_text[end-10:end+20]}")

            
            buffer_rect = b_rect # To be inserted...
            output_blocks.append(new_paragraph_buffer) # It may be empty, if it's a start of a new paragraph, then it will be the symbol
            newline_buffer = False
            bottomright_buffer = None
            # Block text not inserted. Must be inserted manually as we process.
            #txt    = block_text

            output_blocks[-1] += block_text.strip()


            # Append sticky-note comments
            txt = ""
            logger.debug(f"Check Sticky Notes")
            for i, note_data in enumerate(notes):
                n_rect, note, used = note_data
                #logger.debug(f"STICKY NOTE I: {i}, note: {note}, used: {used}")
                if used:
                    continue
                
                if block_cnt == 0 and (b_rect.y0 >= (n_rect.y1+n_rect.y0)/2):
                    logger.debug(f"Sticky note at the top of the page: {note}")
                    notes[i][2] = True
                    #output_blocks[-1] = note_symbol[0]+note.rstrip()+note_symbol[1]+' '+output_blocks[-1]
                    sticky_notes.append({'line': output_blocks[-1][:100], 'local-text': '', 'comment': note, 'context_left': '', 'context_right': output_blocks[-1][:100], 'used': False})
                    continue
                    
                # Pin-point sticky note:
                local_rect = None
                if line_height > 0 and line_height < 1e5:
                    local_rect = fitz.Rect(n_rect.x0-line_height, n_rect.y0-line_height, n_rect.x1+line_height, n_rect.y1+line_height)
                    local_rect = n_rect # It must strictly be on a text
                else:
                    local_rect = n_rect
                on_text = page.get_text("text", clip=local_rect).strip()
                
                if len(on_text)>2 and b_rect.intersects(n_rect) and not used:
                    logger.debug(f"Pin Point needed. Sticky note: {note} on text: {on_text}")
                    # Find which line is it on
                    y_mean = (n_rect.y0+n_rect.y1)/2
                    coef_line = (y_mean - b_rect.y0)//(line_height)
                    logger.debug(f"coef_line: {coef_line}, line_height: {line_height}")
                    y0_tmp = b_rect.y0 + coef_line*line_height
                    y1_tmp = b_rect.y0 + (coef_line+1)*line_height
                    line_text = page.get_text("text", clip=fitz.Rect(b_rect.x0, y0_tmp, b_rect.x1, y1_tmp)).strip()

                    # Instead, Get the text, after seeing coef_line breaklines, that's our line!
                    if int(coef_line) < len(block_text.split('\n')): # Can apply the better method
                        logger.info(f"Applying better method to find the line is: {line_text}")
                        line_text = block_text.split('\n')[int(coef_line)].strip()

                    logger.debug(f"Line text: {line_text}")
                    if line_text:
                        # Find the local text
                        #x0_local = (n_rect.x0 + b_rect.x0) * 2/3
                        first_local = False # First we should insert the local text then the note
                        x0_local = n_rect.x0
                        x1_local = abs(n_rect.x1 - b_rect.x1) * 1/3 + n_rect.x1
                        local_text = page.get_text("text", clip=fitz.Rect(x0_local, y0_tmp, x1_local, y1_tmp)).strip()
                        logger.debug(f"right Local text: {local_text}")
                        if not local_text or len(local_text) < 4:
                            logger.info(f"The right side of the sticky note is empty. Try the left side")
                        else:
                            first_local = True
                        
                        if not first_local:
                            x0_local = n_rect.x0 - abs(n_rect.x0 - b_rect.x0) * 1/3 
                            x1_local = n_rect.x1
                            local_text = page.get_text("text", clip=fitz.Rect(x0_local, y0_tmp, x1_local, y1_tmp)).strip()
                            logger.debug(f"left Local text: {local_text}")
                            if not local_text:
                                logger.warning(f"local text not found on the left side of the sticky note")
                                continue
                        
                        notes[i][2] = True  # Mark this note as used
                        
                        if first_local:
                            sticky_notes.append({'line': line_text, 'local-text': local_text, 'comment': note, 'context_left': '', 'context_right': local_text, 'used': False})
                            continue
                        else:
                            sticky_notes.append({'line': line_text, 'local-text': local_text, 'comment': note, 'context_left': local_text, 'context_right': '', 'used': False})
                            continue

                
                includes = (b_rect.y0 <= (n_rect.y0+n_rect.y1)/2) and (b_rect.y1 >= (n_rect.y0+n_rect.y1)/2)

                line_tmp = ''
                bypass_tmp = False
                
                if block_cnt > 0 and \
                    (n_rect.y0+n_rect.y1)/2 < b_rect.y0 and \
                    (n_rect.y0+n_rect.y1)/2 > blocks[block_cnt-1][3] and \
                    abs((n_rect.y0+n_rect.y1)/2 - b_rect.y0) < abs((n_rect.y0+n_rect.y1)/2 - blocks[block_cnt-1][3]):
                    logger.debug(f"Sticky note:\n{note}\n at the top of the block note, and closer to this block.")
                    notes[i][2] = True
                    line_tmp = output_blocks[-1][:100]
                    sticky_notes.append({'line': line_tmp, 'local-text': '', 'comment': note, 'context_left': '', 'context_right': line_tmp, 'used': False})
                    continue
                
                if block_cnt < len(blocks)-1 and \
                    (n_rect.y0+n_rect.y1)/2 > b_rect.y1 and \
                    (n_rect.y0+n_rect.y1)/2 < blocks[block_cnt+1][1] and \
                    abs((n_rect.y0+n_rect.y1)/2 - b_rect.y1) < abs((n_rect.y0+n_rect.y1)/2 - blocks[block_cnt+1][1]):
                    logger.debug(f"Sticky note:\n{note}\n at the bottom of the block note, and closer to this block. BYPASS if statement and add it:")
                    notes[i][2] = True
                    line_tmp = output_blocks[-1].replace(new_parag_symbol, '')[-100:]
                    bypass_tmp = True

                if (b_rect.intersects(n_rect)) or includes or bypass_tmp:
                    if not line_tmp:
                        line_tmp = output_blocks[-1].replace(new_parag_symbol, '')[-100:] if output_blocks[-1] else ''
                    txt += f"{note}"
                    logger.debug(f"Sticky Note includes: {includes} and does intersect. txt: {txt}")
                    notes[i][2] = True
                    logger.debug(f"Buffer is empty. Sticky notes getting added directly to the output_blocks")
                    #output_blocks[-1] += note_symbol[0]+txt.rstrip()+note_symbol[1]
                    sticky_notes.append({'line': line_tmp, 'local-text': '', 'comment': note, 'context_left': line_tmp, 'context_right': '', 'used': False})
                    

    logger.info(f"Remove the names and dates and junks")
    date_name_block_ind.sort(reverse=True)
    for ind in date_name_block_ind:
        logger.info(f"Removing block: preview: {output_blocks[ind][:20]}...")
        del output_blocks[ind]
    
    result = "\n\n".join(output_blocks)
    logger.info(f"....CALLING PARAGRAPHIZE ON:...\n\n{result}")
    #result = paragraphize_v1(text=result, paragraphs=page_paragraphs, new_page_symbol=new_page_symbol, logger=logger, comment_symbol=comment_symbol, note_symbol=note_symbol, highlight_symbol=highlight_symbol)
    result = paragraphize(text=result, new_page_symbol=new_page_symbol, logger=logger, new_paragraph_symbol=new_parag_symbol)
    
    
    result = combine_paragraphs(result, section_list=section_list, highlight_symbol=highlight_symbol, comment_symbol=comment_symbol, note_symbol=note_symbol, annot_conf_symbol=annot_conf_symbol, logger=logger)

    logger.info(f"Normalizing the sticky notes (removing the paragraph symbol)")
    for sticky_note in sticky_notes:
        sticky_note['line'] = sticky_note['line'].replace(new_parag_symbol, '')
        sticky_note['local-text'] = sticky_note['local-text'].replace(new_parag_symbol, '')

        sticky_note['line'] = sticky_note['line'].replace('\n', ' ')
        sticky_note['context_right'] = re.sub(
        r' +', ' ',
        sticky_note['context_right'].replace('\n', ' ')
        )
        sticky_note['context_left'] = re.sub(
        r' +', ' ',
        sticky_note['context_left'].replace('\n', ' ')
        )
        sticky_note['local-text'] = re.sub(
        r' +', ' ',
        sticky_note['local-text'].replace('\n', ' ')
        )
    
    for highlight in highlights:
        highlight['h_text'] = highlight['h_text'].replace(new_parag_symbol, '')
        highlight['h_text'] = re.sub(
            r' +', ' ',
            highlight['h_text'].replace('\n', ' ')
        )
        highlight['context_left'] = re.sub(
            r' +', ' ',
            highlight['context_left'].replace('\n', ' ')
        )
        highlight['context_right'] = re.sub(
            r' +', ' ',
            highlight['context_right'].replace('\n', ' ')
        )

    logger.info(f"SECTION LIST:  {section_list}")
    logger.info(f'results:  {result}')
    logger.info(f"highlights:  {highlights}")
    logger.info(f"sticky_notes:  {sticky_notes}")
    result = combine_to_json(paragraphs=result, highlights=highlights, sticky_notes=sticky_notes, logger=logger)

    return result


# Run

## Run on one

In [ ]:
pdf_path = "input.pdf"

full_text = parse_pdf(pdf_path)

output_path = 'output.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(full_text, f, ensure_ascii=False, indent=2)

print(full_text)